# 🏠 House Price Prediction — EDA & Model Training

**Dataset:** California Housing Dataset (sklearn)  
**Models:** Linear Regression, Decision Tree, Random Forest, XGBoost  
**Target:** Predict house price (INR scale)

---

## 1. Imports & Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ Libraries loaded successfully')

## 2. Load Dataset

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

df.columns = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
              'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedHouseVal']

# Add synthetic features
np.random.seed(42)
n = len(df)

df['Bedrooms']   = np.clip(np.round(df['AveBedrms'] * 2).astype(int), 1, 6)
df['Bathrooms']  = np.clip(np.round(df['Bedrooms'] * 0.7).astype(int), 1, 4)
df['SquareFeet'] = np.clip((df['AveRooms'] * 200 + np.random.normal(0, 100, n)).astype(int), 500, 5000)
df['YearBuilt']  = np.clip((1950 + df['HouseAge'] + np.random.randint(-5, 5, n)).astype(int), 1900, 2025)
df['Floors']     = np.random.choice([1, 2, 3], size=n, p=[0.4, 0.45, 0.15])
df['Garage']     = np.random.choice([0, 1], size=n, p=[0.3, 0.7])
df['Pool']       = np.random.choice([0, 1], size=n, p=[0.8, 0.2])
df['Location']   = np.random.choice(['Urban', 'Suburban', 'Rural'], size=n, p=[0.5, 0.35, 0.15])
df['Price']      = (df['MedHouseVal'] * 1_000_000).astype(int)

print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('📊 Shape:', df.shape)
df.describe().round(2)

In [ ]:
print('🔍 Null Value Check:')
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else '✅ No nulls found!')

print('\n📋 Data Types:')
print(df.dtypes)

In [ ]:
# Correlation Heatmap
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Price Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Price'], bins=60, color='#f59e0b', alpha=0.8, edgecolor='#d97706')
axes[0].set_title('Price Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price (₹)')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(df['Price']), bins=60, color='#3b82f6', alpha=0.8, edgecolor='#2563eb')
axes[1].set_title('Log(Price) Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Log Price')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Price by Location
plt.figure(figsize=(8, 5))
location_order = ['Urban', 'Suburban', 'Rural']
sns.boxplot(data=df, x='Location', y='Price', order=location_order,
            palette=['#f59e0b', '#3b82f6', '#22c55e'])
plt.title('House Price by Location', fontsize=13, fontweight='bold')
plt.ylabel('Price (₹)')
plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
feature_cols = ['Bedrooms', 'Bathrooms', 'SquareFeet', 'YearBuilt',
                'Floors', 'Garage', 'Pool', 'Location']

df_model = df[feature_cols + ['Price']].copy()

# Handle nulls
for col in df_model.select_dtypes(include=[np.number]).columns:
    df_model[col].fillna(df_model[col].median(), inplace=True)
for col in df_model.select_dtypes(include=['object']).columns:
    df_model[col].fillna(df_model[col].mode()[0], inplace=True)

# Encode Location
le = LabelEncoder()
df_model['Location_Encoded'] = le.fit_transform(df_model['Location'])
print('Location mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

X = df_model[['Bedrooms', 'Bathrooms', 'SquareFeet', 'YearBuilt',
              'Floors', 'Garage', 'Pool', 'Location_Encoded']]
y = df_model['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

## 5. Model Training & Evaluation

In [ ]:
models = {
    'Linear Regression': (LinearRegression(), True),
    'Decision Tree':     (DecisionTreeRegressor(max_depth=12, random_state=42), False),
    'Random Forest':     (RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1), False),
    'XGBoost':           (XGBRegressor(n_estimators=200, max_depth=8, learning_rate=0.1, random_state=42, verbosity=0), False),
}

results = []
predictions = {}

for name, (model, use_scaler) in models.items():
    Xtr = X_train_scaled if use_scaler else X_train
    Xte = X_test_scaled  if use_scaler else X_test
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    predictions[name] = y_pred

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    results.append({'Model': name, 'MAE': f'₹{mae:,.0f}', 'RMSE': f'₹{rmse:,.0f}', 'R²': round(r2, 4)})

results_df = pd.DataFrame(results).set_index('Model')
print('\n📊 Model Comparison:')
results_df

## 6. Actual vs Predicted Plots

In [ ]:
colors = ['#3b82f6', '#10b981', '#f59e0b', '#ef4444']
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('Actual vs Predicted House Prices', fontsize=18, fontweight='bold')

for idx, (name, y_pred) in enumerate(predictions.items()):
    ax  = axes[idx // 2, idx % 2]
    r2  = r2_score(y_test, y_pred)
    ax.scatter(y_test, y_pred, alpha=0.3, s=8, color=colors[idx])
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect Fit')
    ax.set_xlabel('Actual (₹)')
    ax.set_ylabel('Predicted (₹)')
    ax.set_title(f'{name}  |  R² = {r2:.4f}', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Save Best Model

In [ ]:
r2_scores = {name: r2_score(y_test, pred) for name, pred in predictions.items()}
best_name = max(r2_scores, key=r2_scores.get)
print(f'🏆 Best Model: {best_name} (R² = {r2_scores[best_name]:.4f})')

best_model_obj = models[best_name][0]
joblib.dump(best_model_obj, '../model/house_price_model.pkl')
joblib.dump(scaler,         '../model/scaler.pkl')
joblib.dump(le,             '../model/label_encoder.pkl')

metadata = {
    'best_model': best_name,
    'r2_score':   r2_scores[best_name],
    'features':   list(X.columns),
    'location_mapping': dict(zip(le.classes_, le.transform(le.classes_).tolist())),
    'uses_scaler': models[best_name][1]
}
joblib.dump(metadata, '../model/model_metadata.pkl')
print('✅ Model, scaler, encoder, and metadata saved to ../model/')

## 8. Sample Prediction

In [ ]:
sample = {
    'bedrooms': 3, 'bathrooms': 2, 'sqft': 1500,
    'year_built': 2010, 'floors': 1,
    'garage': 1, 'pool': 0, 'location': 'Urban'
}

loc_enc = le.transform([sample['location']])[0]
features = np.array([[sample['bedrooms'], sample['bathrooms'], sample['sqft'],
                      sample['year_built'], sample['floors'],
                      sample['garage'], sample['pool'], loc_enc]])

if metadata['uses_scaler']:
    features = scaler.transform(features)

price = best_model_obj.predict(features)[0]
print(f'Input: {sample}')
print(f'Predicted Price : ₹{price:,.0f}')
print(f'Range (±10%)    : ₹{price*0.9:,.0f}  –  ₹{price*1.1:,.0f}')
print(f'Confidence      : {"High" if r2_scores[best_name] >= 0.85 else "Medium"}')